In [1]:
import os, torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))
print("cuda available =", torch.cuda.is_available())
print("device count =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0 =", torch.cuda.get_device_name(0))

CUDA_VISIBLE_DEVICES = 1
LD_LIBRARY_PATH = None
cuda available = True
device count = 1
device 0 = NVIDIA RTX A6000


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import csv
import json
import math
import time
import uuid
import random
import hashlib
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple, Optional, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm import HyperbolicLCM


@dataclass
class HLCMGSM8KHardNegGRPOConfig:
    # data
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet"

    # outputs
    out_dir: str = "runs/hlcm_gsm8k"
    cache_dir: str = "gsm8k_hardneg_cache"
    ref_logits_dir: str = "gsm8k_ref_logits"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"
    use_normalizer: bool = False

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # HLCM arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.10
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune
    finetune_mode: str = "last_blocks" 
    n_last_blocks: int = 1

    # ranking setup
    num_choices: int = 8
    min_valid_choices: int = 4
    choice_chunk_size: int = 2
    use_hard_negatives: bool = True
    hard_negative_pool_from_train_only: bool = True
    hard_negative_close_k: int = 64

    # prompt
    instruction: str = (
        "Solve the grade-school math word problem.\n"
        "Return only the final numeric answer."
    )

    # optimization
    train_batch_size: int = 4
    eval_batch_size: int = 8
    grad_accum_steps: int = 2
    num_workers: int = 0
    max_grad_norm: float = 1.0
    weight_decay: float = 0.01
    use_bf16: bool = True

    # SFT stage
    sft_epochs: int = 12
    sft_lr: float = 5e-5
    sft_warmup_ratio: float = 0.03

    # GRPO stage
    run_grpo: bool = True
    grpo_epochs: int = 4
    grpo_lr: float = 1e-5
    grpo_warmup_ratio: float = 0.03
    grpo_group_size: int = 8
    grpo_beta_kl: float = 0.02
    grpo_policy_temperature: float = 1.0
    mcq_logit_temperature: float = 0.1
    entropy_bonus: float = 0.001
    use_group_relative_advantage: bool = True

    # rewards
    reward_correct: float = 1.0
    reward_incorrect: float = 0.0

    # stability
    clamp_tangent_value: float = 100.0
    replace_nonfinite_with_zero: bool = True
    strict_finite_checks: bool = False

    # checkpointing
    save_last_every_epoch: bool = False
    max_prediction_files_to_keep: int = 2

    seed: int = 42
    prefer_gpu_index: int = 0


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def safe_json_dump(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + f".tmp.{uuid.uuid4().hex}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def prune_prediction_files(out_dir: str, prefix: str, keep: int):
    files = []
    for fn in os.listdir(out_dir):
        if fn.startswith(prefix) and fn.endswith(".json"):
            full = os.path.join(out_dir, fn)
            files.append((os.path.getmtime(full), full))
    files.sort()
    if len(files) > keep:
        for _, path in files[:-keep]:
            try:
                os.remove(path)
            except Exception:
                pass


def assert_finite(name: str, x: torch.Tensor):
    if not torch.isfinite(x).all():
        bad = (~torch.isfinite(x)).sum().item()
        raise RuntimeError(f"{name} has non-finite values; bad_count={bad}; shape={tuple(x.shape)}")


def sanitize_tensor(x: torch.Tensor, clamp_value: float, replace_nonfinite_with_zero: bool) -> torch.Tensor:
    if replace_nonfinite_with_zero:
        x = torch.nan_to_num(x, nan=0.0, posinf=clamp_value, neginf=-clamp_value)
    x = torch.clamp(x, -clamp_value, clamp_value)
    return x


def load_normalizer(normalizer_path: str, device: torch.device, use_normalizer: bool):
    if not use_normalizer:
        return None, None
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def stable_int_hash(text: str) -> int:
    h = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(h[:16], 16)


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + f".tmp.{uuid.uuid4().hex}"
    torch.save(payload, tmp)
    os.replace(tmp, path)


def safe_float_from_fraction_or_decimal(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    if not s:
        return None

    try:
        return float(s)
    except Exception:
        pass

    frac_match = re.fullmatch(r"[-+]?\d+\s*/\s*\d+", s)
    if frac_match:
        try:
            from fractions import Fraction
            return float(Fraction(s.replace(" ", "")))
        except Exception:
            return None
    return None


def canonicalize_numeric_str(s: str) -> str:
    s = str(s).strip().replace(",", "")
    if s == "":
        return ""

    val = safe_float_from_fraction_or_decimal(s)
    if val is not None and math.isfinite(val):
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return f"{val:.8f}".rstrip("0").rstrip(".")
    return s


def extract_final_numeric_answer(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""

    if "####" in text:
        candidate = text.split("####")[-1].strip()
        return canonicalize_numeric_str(candidate)

    frac_matches = re.findall(r"[-+]?\d+\s*/\s*\d+", text)
    dec_matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    all_candidates = frac_matches + dec_matches

    if all_candidates:
        best = None
        best_pos = -1
        for m in all_candidates:
            pos = text.rfind(m)
            if pos > best_pos:
                best = m
                best_pos = pos
        return canonicalize_numeric_str(best)

    return canonicalize_numeric_str(text)


def numeric_equal(a: str, b: str, tol: float = 1e-6) -> bool:
    fa = safe_float_from_fraction_or_decimal(a)
    fb = safe_float_from_fraction_or_decimal(b)
    if fa is not None and fb is not None:
        return abs(fa - fb) <= tol
    return canonicalize_numeric_str(a) == canonicalize_numeric_str(b)


def try_parse_float(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    try:
        x = float(s)
        if math.isfinite(x):
            return float(x)
    except Exception:
        return None
    return None


def is_integerish_str(s: str) -> bool:
    x = try_parse_float(s)
    return (x is not None) and float(x).is_integer()


def same_sign(a: Optional[float], b: Optional[float]) -> bool:
    if a is None or b is None:
        return False
    if a == 0.0 and b == 0.0:
        return True
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def num_decimal_places(s: str) -> int:
    s = canonicalize_numeric_str(s)
    if "." not in s:
        return 0
    return len(s.split(".")[-1])


def normalize_local_gsm8k_row(row: Dict[str, Any]) -> Dict[str, Any]:
    row = dict(row)

    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()

    final_numeric_answer = extract_final_numeric_answer(answer)

    row["question"] = question
    row["answer"] = answer
    row["final_numeric_answer"] = final_numeric_answer
    return row


def load_gsm8k_local(cfg: HLCMGSM8KHardNegGRPOConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )
    return raw["train"], raw["validation"]


def build_gsm8k_prompt(question: str, instruction: str) -> str:
    return "\n".join([
        instruction.strip(),
        "",
        "Problem:",
        question.strip(),
        "",
        "Final Answer:",
    ])


def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")

    n_last = max(1, min(n_last, len(layers)))
    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                hs = self.enc(**inputs).last_hidden_state
        else:
            hs = self.enc(**inputs).last_hidden_state

        attn = inputs["attention_mask"].unsqueeze(-1).float()
        summed = (hs * attn).sum(dim=1)
        denom = attn.sum(dim=1).clamp_min(1.0)
        out = summed / denom
        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


def collect_answer_pool(hf_split) -> List[str]:
    pool: Set[str] = set()
    for ex in hf_split:
        ex = normalize_local_gsm8k_row(ex)
        ans = ex["final_numeric_answer"]
        if ans:
            pool.add(ans)
    out = sorted(pool)
    if len(out) < 2:
        raise ValueError("Answer pool is too small to build negative candidates.")
    return out


def build_answer_pool_metadata(answer_pool: List[str]) -> List[Dict[str, Any]]:
    meta = []
    for ans in answer_pool:
        x = try_parse_float(ans)
        meta.append({
            "answer": ans,
            "value": x,
            "abs_value": abs(x) if x is not None else float("inf"),
            "is_integerish": is_integerish_str(ans),
            "decimals": num_decimal_places(ans),
            "str_len": len(ans),
        })
    return meta


def deterministic_fallback_order_key(candidate_answer: str, gold_answer: str, question: str) -> Tuple[int, str]:
    seed_text = f"{question} || {gold_answer} || {candidate_answer}"
    return (stable_int_hash(seed_text), candidate_answer)


def choose_hard_negative_answers(
    gold_answer: str,
    question: str,
    answer_pool_meta: List[Dict[str, Any]],
    k_neg: int,
    close_k: int = 64,
) -> List[str]:
    gold_value = try_parse_float(gold_answer)
    gold_is_int = is_integerish_str(gold_answer)
    gold_decimals = num_decimal_places(gold_answer)
    gold_len = len(gold_answer)

    candidates = []
    for item in answer_pool_meta:
        cand = item["answer"]
        if cand == gold_answer:
            continue

        val = item["value"]
        dist = abs(val - gold_value) if gold_value is not None and val is not None else float("inf")
        same_sign_flag = 1 if same_sign(gold_value, val) else 0
        same_int_flag = 1 if item["is_integerish"] == gold_is_int else 0
        same_dec_flag = 1 if item["decimals"] == gold_decimals else 0
        len_gap = abs(item["str_len"] - gold_len)
        abs_gap = abs(item["abs_value"] - abs(gold_value)) if gold_value is not None and val is not None else float("inf")
        fallback_hash, _ = deterministic_fallback_order_key(cand, gold_answer, question)

        candidates.append({
            "answer": cand,
            "dist": dist,
            "same_sign": same_sign_flag,
            "same_int": same_int_flag,
            "same_dec": same_dec_flag,
            "len_gap": len_gap,
            "abs_gap": abs_gap,
            "fallback_hash": fallback_hash,
        })

    candidates.sort(
        key=lambda z: (
            z["dist"],
            -z["same_sign"],
            -z["same_int"],
            -z["same_dec"],
            z["abs_gap"],
            z["len_gap"],
            z["fallback_hash"],
            z["answer"],
        )
    )

    close_candidates = candidates[:max(k_neg * 4, close_k)]
    if len(close_candidates) == 0:
        raise ValueError("No negative answers available.")

    selected = []
    seen = set()

    def add_if_new(ans: str):
        if ans not in seen and ans != gold_answer:
            selected.append(ans)
            seen.add(ans)

    for item in close_candidates:
        add_if_new(item["answer"])
        if len(selected) >= k_neg:
            return selected[:k_neg]

    for item in candidates:
        add_if_new(item["answer"])
        if len(selected) >= k_neg:
            return selected[:k_neg]

    return selected[:k_neg]


def cache_file_path(cfg: HLCMGSM8KHardNegGRPOConfig, split_name: str) -> str:
    ensure_dir(cfg.cache_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"
    return os.path.join(
        cfg.cache_dir,
        f"{split_name}_{hard_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def ref_logits_file_path(cfg: HLCMGSM8KHardNegGRPOConfig, split_name: str) -> str:
    ensure_dir(cfg.ref_logits_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"
    return os.path.join(
        cfg.ref_logits_dir,
        f"{split_name}_{hard_tag}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def normalize_gsm8k_example_for_ranking(
    ex: Dict[str, Any],
    cfg: HLCMGSM8KHardNegGRPOConfig,
    answer_pool_meta: List[Dict[str, Any]],
) -> Tuple[str, List[str], int, str]:
    ex = normalize_local_gsm8k_row(ex)
    question = ex["question"]
    gold_answer = ex["final_numeric_answer"]

    if not question:
        raise ValueError("Empty question.")
    if not gold_answer:
        raise ValueError("Empty final numeric answer.")

    q_text = build_gsm8k_prompt(question, cfg.instruction)

    k_total = max(cfg.min_valid_choices, cfg.num_choices)
    k_neg = max(1, k_total - 1)

    negatives = choose_hard_negative_answers(
        gold_answer=gold_answer,
        question=question,
        answer_pool_meta=answer_pool_meta,
        k_neg=k_neg,
        close_k=cfg.hard_negative_close_k,
    )

    choice_texts = [gold_answer] + negatives
    label = 0
    return q_text, choice_texts, label, gold_answer


def build_or_load_cached_split(
    cfg: HLCMGSM8KHardNegGRPOConfig,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
    answer_pool_meta: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label, gold_answer = normalize_gsm8k_example_for_ranking(
                ex=ex,
                cfg=cfg,
                answer_pool_meta=answer_pool_meta,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)
            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"{q_text} {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)
            choices = torch.stack(c_seqs, dim=0)
            cmask = torch.stack(c_pads, dim=0)
            choice_mask = torch.ones(K, dtype=torch.bool)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": choices,
                "cmask": cmask,
                "choice_mask": choice_mask,
                "label": int(label),
                "num_choices": int(K),
                "gold_answer": gold_answer,
                "choice_texts": choice_texts,
                "question_text": q_text,
            })
        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows

# DATASET

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
            "gold_answer": r["gold_answer"],
            "choice_texts": r["choice_texts"],
            "question_text": r["question_text"],
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    choice_texts = []
    gold_answers = []
    question_texts = []

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]
        choice_texts.append(item["choice_texts"])
        gold_answers.append(item["gold_answer"])
        question_texts.append(item["question_text"])

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
        "choice_texts": choice_texts,
        "gold_answers": gold_answers,
        "question_texts": question_texts,
    }


def build_hlcm_from_cfg(cfg: HLCMGSM8KHardNegGRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: HLCMGSM8KHardNegGRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


def hlcm_encode_full_memory_safe(model: HyperbolicLCM, x: torch.Tensor) -> torch.Tensor:
    if not hasattr(model, "encode_inputs") or not hasattr(model, "layers"):
        return model(x)

    layers = list(model.layers)
    if len(layers) == 0:
        return model(x)

    first_trainable = len(layers)
    for i, layer in enumerate(layers):
        has_grad = any(p.requires_grad for p in layer.parameters())
        if has_grad:
            first_trainable = i
            break

    if first_trainable <= 0:
        h = model.encode_inputs(x)
        for layer in layers:
            h = layer(h)
        return h

    with torch.no_grad():
        h = model.encode_inputs(x)
        for layer in layers[:first_trainable]:
            h = layer(h)

    h = h.detach()
    for layer in layers[first_trainable:]:
        h = layer(h)

    return h


def hlcm_tangent_sequence(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if cfg.strict_finite_checks:
        assert_finite("input_x_before_norm", x)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    x = sanitize_tensor(x, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h = hlcm_encode_full_memory_safe(model, x)
    h = sanitize_tensor(h, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h_tan = model.manifold.logmap0(h)
    h_tan = sanitize_tensor(h_tan, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)

    if cfg.strict_finite_checks:
        assert_finite("hlcm_h_tan", h_tan)

    return h_tan


def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
) -> torch.Tensor:
    h_tan = hlcm_tangent_sequence(model, x, pad_mask, mu, sigma, cfg)
    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask.to(h_tan.device)[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma, cfg=cfg)
    e_q = F.normalize(e_q, dim=-1)

    logits_list = []
    step = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, step):
        k1 = min(K, k0 + step)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        ec = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma, cfg=cfg)
        ec = ec.reshape(B, (k1 - k0), -1)
        ec = F.normalize(ec, dim=-1)

        chunk_logits = torch.einsum("bd,bkd->bk", e_q, ec)
        logits_list.append(chunk_logits)

    logits = torch.cat(logits_list, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
):
    logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
    y = batch["label"].to(logits.device, non_blocking=True)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)

# EVAL

@torch.no_grad()
def evaluate_hlcm_detailed(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0, "chance_acc": 0.0}

    model.eval()
    tot_loss = 0.0
    tot_acc = 0.0
    tot_chance = 0.0
    n = 0

    for batch in loader:
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        labels = batch["label"].to(logits.device, non_blocking=True)
        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=1)
        bs = labels.size(0)
        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()

        tot_loss += float(loss.item()) * bs
        tot_acc += float((preds == labels).float().sum().item())
        for k in choice_counts:
            tot_chance += 1.0 / max(1, int(k))
        n += bs

    return {
        "loss": tot_loss / max(1, n),
        "acc": tot_acc / max(1, n),
        "chance_acc": tot_chance / max(1, n),
    }


@torch.no_grad()
def dump_eval_predictions(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
    out_path: str,
):
    model.eval()
    rows = []

    for batch in loader:
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        probs = F.softmax(logits, dim=-1)
        pred_idx = logits.argmax(dim=1).detach().cpu().tolist()
        gold_idx = batch["label"].detach().cpu().tolist()
        choice_counts = batch["choice_mask"].sum(dim=1).cpu().tolist()

        for i in range(len(pred_idx)):
            p = pred_idx[i]
            g = gold_idx[i]
            valid_k = int(choice_counts[i])
            logits_i = logits[i, :valid_k].detach().cpu()
            probs_i = probs[i, :valid_k].detach().cpu()
            choices = batch["choice_texts"][i]
            rows.append({
                "question_text": batch["question_texts"][i],
                "gold_answer": batch["gold_answers"][i],
                "pred_answer": choices[p],
                "gold_choice_index": g,
                "pred_choice_index": p,
                "correct": int(p == g),
                "choice_texts": choices,
                "choice_logits": [float(x) for x in logits_i.tolist()],
                "choice_probs": [float(x) for x in probs_i.tolist()],
            })

    safe_json_dump(rows, out_path)


# OPTIMIZER

def make_optimizer_and_scheduler(trainable_params, lr: float, total_steps: int, warmup_ratio: float, weight_decay: float):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError("No trainable parameters found.")

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: HLCMGSM8KHardNegGRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()
    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# SFT

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
    device: torch.device,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler("cuda", enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()))

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] val_loss={base_eval['loss']:.4f} val_acc={base_eval['acc']:.4f} chance={base_eval['chance_acc']:.4f}")
    append_dict_to_csv(eval_csv, {
        "epoch": 0,
        "train_loss": "",
        "val_loss": base_eval["loss"],
        "val_acc": base_eval["acc"],
        "val_chance_acc": base_eval["chance_acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)
    global_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(model, batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)
        eval_metrics = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={eval_metrics['loss']:.4f} val_acc={eval_metrics['acc']:.4f} chance={eval_metrics['chance_acc']:.4f}"
        )

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })
        append_dict_to_csv(eval_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": eval_metrics["loss"],
            "val_acc": eval_metrics["acc"],
            "val_chance_acc": eval_metrics["chance_acc"],
        })

        if eval_metrics["acc"] > best_acc:
            best_acc = eval_metrics["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_val_acc": best_acc,
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        if cfg.save_last_every_epoch:
            save_checkpoint(
                {
                    "model": clone_state_dict_to_cpu(model),
                    "stage": "sft_last",
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "sft_last.pt"),
            )

        cuda_cleanup()

    model.load_state_dict(best_state, strict=True)
    return {"best_acc": best_acc, "best_state": best_state}


# GRPO

@torch.no_grad()
def sample_group_actions(logits: torch.Tensor, group_size: int, policy_temperature: float) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)
    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
    }
    return total_loss, stats


def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: HLCMGSM8KHardNegGRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler("cuda", enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()))

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base_eval = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] val_loss={base_eval['loss']:.4f} val_acc={base_eval['acc']:.4f} chance={base_eval['chance_acc']:.4f}")
    append_dict_to_csv(eval_csv, {
        "epoch": 0,
        "train_loss": "",
        "val_loss": base_eval["loss"],
        "val_acc": base_eval["acc"],
        "val_chance_acc": base_eval["chance_acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)
    global_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)
        epoch_loss_sum = 0.0
        epoch_reward_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)
            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(model, batch, ref_logits_batch, mu, sigma, cfg)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(model, batch, ref_logits_batch, mu, sigma, cfg)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_reward_sum += float(stats["reward_mean"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                reward=f"{epoch_reward_sum / max(1, epoch_count):.4f}",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_reward = epoch_reward_sum / max(1, epoch_count)
        eval_metrics = evaluate_hlcm_detailed(model, eval_loader, mu, sigma, cfg)

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] train_loss={train_loss:.4f} train_reward={train_reward:.4f} "
            f"val_loss={eval_metrics['loss']:.4f} val_acc={eval_metrics['acc']:.4f} chance={eval_metrics['chance_acc']:.4f}"
        )

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_reward": train_reward,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
        })
        append_dict_to_csv(eval_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_reward": train_reward,
            "val_loss": eval_metrics["loss"],
            "val_acc": eval_metrics["acc"],
            "val_chance_acc": eval_metrics["chance_acc"],
        })

        if eval_metrics["acc"] > best_acc:
            best_acc = eval_metrics["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_val_acc": best_acc,
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        if cfg.save_last_every_epoch:
            save_checkpoint(
                {
                    "model": clone_state_dict_to_cpu(model),
                    "stage": "grpo_last",
                    "epoch": epoch,
                    "config": asdict(cfg),
                },
                os.path.join(out_dir, "grpo_last.pt"),
            )

        cuda_cleanup()

    model.load_state_dict(best_state, strict=True)
    return {"best_acc": best_acc, "best_state": best_state}


# MAIN

def main():
    cfg = HLCMGSM8KHardNegGRPOConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Using normalizer:", cfg.use_normalizer)

    train_hf, eval_hf = load_gsm8k_local(cfg)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    if cfg.hard_negative_pool_from_train_only:
        answer_pool = collect_answer_pool(train_hf)
    else:
        answer_pool = sorted(set(collect_answer_pool(train_hf)).union(set(collect_answer_pool(eval_hf))))
    answer_pool_meta = build_answer_pool_metadata(answer_pool)

    print(f"[pool] unique answers in hard-negative pool: {len(answer_pool_meta)}")

    train_rows = build_or_load_cached_split(cfg, "train", train_hf, conceptizer, answer_pool_meta)
    eval_rows = build_or_load_cached_split(cfg, "validation", eval_hf, conceptizer, answer_pool_meta)

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )
    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)

    total_params = sum(p.numel() for p in hlcm.parameters())
    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    print(f"Train rows: {len(train_ds)}")
    print(f"Eval rows: {len(eval_ds)}")

    mu, sigma = load_normalizer(cfg.normalizer_path, device, cfg.use_normalizer)

    print("\nSTAGE 1: SFT")
    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=cfg.out_dir,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "sft_best_acc": sft_result["best_acc"],
            "config": asdict(cfg),
        },
        os.path.join(cfg.out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()

    grpo_result = None
    if cfg.run_grpo:
        ref_logits_path = ref_logits_file_path(cfg, "train")
        ref_logits_rows = precompute_reference_logits(
            model=hlcm,
            dataset=train_ds,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
            out_path=ref_logits_path,
        )
        cuda_cleanup()

        print("\nSTAGE 2: GRPO")
        grpo_result = run_stage_grpo_hlcm_cached_ref(
            model=hlcm,
            train_loader=train_loader,
            eval_loader=eval_loader,
            ref_logits_rows=ref_logits_rows,
            out_dir=cfg.out_dir,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            device=device,
        )

        hlcm.load_state_dict(grpo_result["best_state"], strict=True)
        del grpo_result["best_state"]
        cuda_cleanup()

    final_eval = evaluate_hlcm_detailed(hlcm, eval_loader, mu, sigma, cfg)
    dump_eval_predictions(
        model=hlcm,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        out_path=os.path.join(cfg.out_dir, "final_eval_predictions.json"),
    )

    summary = {
        "sft_best_acc": None if 'sft_result' not in locals() else sft_result["best_acc"],
        "grpo_best_acc": None if grpo_result is None else grpo_result["best_acc"],
        "final_val_loss": final_eval["loss"],
        "final_val_acc": final_eval["acc"],
        "final_val_chance_acc": final_eval["chance_acc"],
        "config": asdict(cfg),
    }
    safe_json_dump(summary, os.path.join(cfg.out_dir, "final_summary.json"))

    print("\nDone.")
    print(f"SFT best acc:  {summary['sft_best_acc']:.4f}")
    if grpo_result is not None:
        print(f"GRPO best acc: {summary['grpo_best_acc']:.4f}")
    print(f"Final val acc: {summary['final_val_acc']:.4f}")
    print("Outputs in:", cfg.out_dir)


if __name__ == "__main__":
    main()

Device: cuda:0
Using normalizer: False


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

[GRPO][epoch 2/4] train_loss=-0.0019 train_reward=0.1238 val_loss=2.0794 val_acc=0.1948 chance=0.1250


grpo 3/4: 100%|█| 1869/1869 [26:37<00:00,  1.17it/s, kl=0.0020, loss=-0.0018, lr=1.55e-06, reward=0.


[GRPO][epoch 3/4] train_loss=-0.0018 train_reward=0.1247 val_loss=2.0794 val_acc=0.1941 chance=0.1250


grpo 4/4: 100%|█| 1869/1869 [26:36<00:00,  1.17it/s, kl=0.0017, loss=-0.0021, lr=0.00e+00, reward=0.


[GRPO][epoch 4/4] train_loss=-0.0021 train_reward=0.1265 val_loss=2.0794 val_acc=0.1941 chance=0.1250

Done.
SFT best acc:  0.1956
GRPO best acc: 0.1956
Final val acc: 0.1956
Outputs in: runs/hlcm_gsm8k


In [1]:
# ============================================================
# GSM8K after_sft.pt EVAL ONLY
# Metrics: Loss, Accuracy, Precision/Recall/F1,
# Precision@k, Recall@k, MRR, Brier score, ECE, MCE
# ============================================================

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import csv
import json
import math
import uuid
import random
import hashlib
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple, Optional, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class EvalConfig:
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_gsm8k"
    cache_dir: str = "gsm8k_hardneg_cache"

    base_ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    eval_ckpt_path: str = "runs/hlcm_gsm8k/after_sft.pt"

    normalizer_path: str = "normalizer.pt"
    use_normalizer: bool = False

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.10
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    num_choices: int = 8
    min_valid_choices: int = 4
    choice_chunk_size: int = 2
    use_hard_negatives: bool = True
    hard_negative_pool_from_train_only: bool = True
    hard_negative_close_k: int = 64

    instruction: str = (
        "Solve the grade-school math word problem.\n"
        "Return only the final numeric answer."
    )

    eval_batch_size: int = 8
    num_workers: int = 0
    mcq_logit_temperature: float = 0.1

    clamp_tangent_value: float = 100.0
    replace_nonfinite_with_zero: bool = True
    strict_finite_checks: bool = False

    ece_bins: int = 15

    seed: int = 42
    prefer_gpu_index: int = 0
    use_bf16: bool = True


cfg = EvalConfig()


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_json_dump(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + f".tmp.{uuid.uuid4().hex}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def load_normalizer(path: str, device: torch.device, use_normalizer: bool):
    if not use_normalizer:
        return None, None
    if not path or not os.path.exists(path):
        return None, None
    obj = torch.load(path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def assert_finite(name: str, x: torch.Tensor):
    if not torch.isfinite(x).all():
        bad = (~torch.isfinite(x)).sum().item()
        raise RuntimeError(f"{name} has non-finite values; bad_count={bad}; shape={tuple(x.shape)}")


def sanitize_tensor(x: torch.Tensor, clamp_value: float, replace_nonfinite_with_zero: bool) -> torch.Tensor:
    if replace_nonfinite_with_zero:
        x = torch.nan_to_num(x, nan=0.0, posinf=clamp_value, neginf=-clamp_value)
    return torch.clamp(x, -clamp_value, clamp_value)


def stable_int_hash(text: str) -> int:
    h = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(h[:16], 16)


# ============================================================
# NUMERIC UTILS
# ============================================================

def safe_float_from_fraction_or_decimal(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    if not s:
        return None

    try:
        return float(s)
    except Exception:
        pass

    if re.fullmatch(r"[-+]?\d+\s*/\s*\d+", s):
        try:
            from fractions import Fraction
            return float(Fraction(s.replace(" ", "")))
        except Exception:
            return None
    return None


def canonicalize_numeric_str(s: str) -> str:
    s = str(s).strip().replace(",", "")
    if s == "":
        return ""

    val = safe_float_from_fraction_or_decimal(s)
    if val is not None and math.isfinite(val):
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return f"{val:.8f}".rstrip("0").rstrip(".")
    return s


def extract_final_numeric_answer(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""

    if "####" in text:
        return canonicalize_numeric_str(text.split("####")[-1].strip())

    frac_matches = re.findall(r"[-+]?\d+\s*/\s*\d+", text)
    dec_matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
    all_candidates = frac_matches + dec_matches

    if all_candidates:
        best = None
        best_pos = -1
        for m in all_candidates:
            pos = text.rfind(m)
            if pos > best_pos:
                best = m
                best_pos = pos
        return canonicalize_numeric_str(best)

    return canonicalize_numeric_str(text)


def try_parse_float(s: str) -> Optional[float]:
    s = str(s).strip().replace(",", "")
    try:
        x = float(s)
        if math.isfinite(x):
            return float(x)
    except Exception:
        return None
    return None


def is_integerish_str(s: str) -> bool:
    x = try_parse_float(s)
    return (x is not None) and float(x).is_integer()


def same_sign(a: Optional[float], b: Optional[float]) -> bool:
    if a is None or b is None:
        return False
    if a == 0.0 and b == 0.0:
        return True
    return (a > 0 and b > 0) or (a < 0 and b < 0)


def num_decimal_places(s: str) -> int:
    s = canonicalize_numeric_str(s)
    if "." not in s:
        return 0
    return len(s.split(".")[-1])


# ============================================================
# DATA
# ============================================================

def normalize_local_gsm8k_row(row: Dict[str, Any]) -> Dict[str, Any]:
    row = dict(row)
    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()
    row["question"] = question
    row["answer"] = answer
    row["final_numeric_answer"] = extract_final_numeric_answer(answer)
    return row


def load_gsm8k_local(cfg: EvalConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )
    return raw["train"], raw["validation"]


def build_gsm8k_prompt(question: str, instruction: str) -> str:
    return "\n".join([
        instruction.strip(),
        "",
        "Problem:",
        question.strip(),
        "",
        "Final Answer:",
    ])


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name: str, chunk_tok_len: int, seq_len: int, batch_size: int, device: torch.device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                hs = self.enc(**inputs).last_hidden_state
        else:
            hs = self.enc(**inputs).last_hidden_state

        attn = inputs["attention_mask"].unsqueeze(-1).float()
        out = (hs * attn).sum(dim=1) / attn.sum(dim=1).clamp_min(1.0)
        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# HARD NEGATIVE ANSWER POOL
# ============================================================

def collect_answer_pool(hf_split) -> List[str]:
    pool: Set[str] = set()
    for ex in hf_split:
        ex = normalize_local_gsm8k_row(ex)
        ans = ex["final_numeric_answer"]
        if ans:
            pool.add(ans)
    out = sorted(pool)
    if len(out) < 2:
        raise ValueError("Answer pool is too small.")
    return out


def build_answer_pool_metadata(answer_pool: List[str]) -> List[Dict[str, Any]]:
    meta = []
    for ans in answer_pool:
        x = try_parse_float(ans)
        meta.append({
            "answer": ans,
            "value": x,
            "abs_value": abs(x) if x is not None else float("inf"),
            "is_integerish": is_integerish_str(ans),
            "decimals": num_decimal_places(ans),
            "str_len": len(ans),
        })
    return meta


def deterministic_fallback_order_key(candidate_answer: str, gold_answer: str, question: str) -> Tuple[int, str]:
    seed_text = f"{question} || {gold_answer} || {candidate_answer}"
    return stable_int_hash(seed_text), candidate_answer


def choose_hard_negative_answers(
    gold_answer: str,
    question: str,
    answer_pool_meta: List[Dict[str, Any]],
    k_neg: int,
    close_k: int = 64,
) -> List[str]:
    gold_value = try_parse_float(gold_answer)
    gold_is_int = is_integerish_str(gold_answer)
    gold_decimals = num_decimal_places(gold_answer)
    gold_len = len(gold_answer)

    candidates = []

    for item in answer_pool_meta:
        cand = item["answer"]
        if cand == gold_answer:
            continue

        val = item["value"]
        dist = abs(val - gold_value) if gold_value is not None and val is not None else float("inf")
        same_sign_flag = 1 if same_sign(gold_value, val) else 0
        same_int_flag = 1 if item["is_integerish"] == gold_is_int else 0
        same_dec_flag = 1 if item["decimals"] == gold_decimals else 0
        len_gap = abs(item["str_len"] - gold_len)
        abs_gap = abs(item["abs_value"] - abs(gold_value)) if gold_value is not None and val is not None else float("inf")
        fallback_hash, _ = deterministic_fallback_order_key(cand, gold_answer, question)

        candidates.append({
            "answer": cand,
            "dist": dist,
            "same_sign": same_sign_flag,
            "same_int": same_int_flag,
            "same_dec": same_dec_flag,
            "len_gap": len_gap,
            "abs_gap": abs_gap,
            "fallback_hash": fallback_hash,
        })

    candidates.sort(
        key=lambda z: (
            z["dist"],
            -z["same_sign"],
            -z["same_int"],
            -z["same_dec"],
            z["abs_gap"],
            z["len_gap"],
            z["fallback_hash"],
            z["answer"],
        )
    )

    selected = []
    seen = set()

    for item in candidates[:max(k_neg * 4, close_k)] + candidates:
        ans = item["answer"]
        if ans not in seen and ans != gold_answer:
            selected.append(ans)
            seen.add(ans)
        if len(selected) >= k_neg:
            break

    return selected[:k_neg]


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: EvalConfig, split_name: str) -> str:
    ensure_dir(cfg.cache_dir)
    hard_tag = "hardneg" if cfg.use_hard_negatives else "randneg"
    return os.path.join(
        cfg.cache_dir,
        f"{split_name}_{hard_tag}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}_K{cfg.num_choices}.pt"
    )


def normalize_gsm8k_example_for_ranking(
    ex: Dict[str, Any],
    cfg: EvalConfig,
    answer_pool_meta: List[Dict[str, Any]],
) -> Tuple[str, List[str], int, str]:
    ex = normalize_local_gsm8k_row(ex)
    question = ex["question"]
    gold_answer = ex["final_numeric_answer"]

    if not question:
        raise ValueError("Empty question.")
    if not gold_answer:
        raise ValueError("Empty final numeric answer.")

    q_text = build_gsm8k_prompt(question, cfg.instruction)

    k_total = max(cfg.min_valid_choices, cfg.num_choices)
    k_neg = max(1, k_total - 1)

    negatives = choose_hard_negative_answers(
        gold_answer=gold_answer,
        question=question,
        answer_pool_meta=answer_pool_meta,
        k_neg=k_neg,
        close_k=cfg.hard_negative_close_k,
    )

    choice_texts = [gold_answer] + negatives
    label = 0
    return q_text, choice_texts, label, gold_answer


def build_or_load_cached_split(
    cfg: EvalConfig,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
    answer_pool_meta: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label, gold_answer = normalize_gsm8k_example_for_ranking(
                ex=ex,
                cfg=cfg,
                answer_pool_meta=answer_pool_meta,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs, c_pads = [], []
            for ct in choice_texts:
                qc = f"{q_text} {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(len(c_seqs), dtype=torch.bool),
                "label": int(label),
                "num_choices": int(len(c_seqs)),
                "gold_answer": gold_answer,
                "choice_texts": choice_texts,
                "question_text": q_text,
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
            "gold_answer": r["gold_answer"],
            "choice_texts": r["choice_texts"],
            "question_text": r["question_text"],
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    choice_texts = []
    gold_answers = []
    question_texts = []

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]
        choice_texts.append(item["choice_texts"])
        gold_answers.append(item["gold_answer"])
        question_texts.append(item["question_text"])

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
        "choice_texts": choice_texts,
        "gold_answers": gold_answers,
        "question_texts": question_texts,
    }


# ============================================================
# MODEL LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: EvalConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_base_then_after_sft(cfg: EvalConfig, device: torch.device) -> Tuple[HyperbolicLCM, Dict[str, Any]]:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.base_ckpt_path):
        raise FileNotFoundError(f"Base checkpoint not found: {cfg.base_ckpt_path}")
    if not os.path.exists(cfg.eval_ckpt_path):
        raise FileNotFoundError(f"after_sft checkpoint not found: {cfg.eval_ckpt_path}")

    base_obj = torch.load(cfg.base_ckpt_path, map_location="cpu")
    base_state = base_obj["model"] if isinstance(base_obj, dict) and "model" in base_obj else base_obj
    model.load_state_dict(base_state, strict=False)

    ckpt = torch.load(cfg.eval_ckpt_path, map_location="cpu")
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] after_sft checkpoint: {cfg.eval_ckpt_path}")
    print(f"[load] stage: {ckpt.get('stage', None) if isinstance(ckpt, dict) else None}")
    print(f"[load] missing={len(missing)}, unexpected={len(unexpected)}")

    model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model, ckpt if isinstance(ckpt, dict) else {}


# ============================================================
# HLCM FORWARD / LOGITS
# ============================================================

def hlcm_encode_full_memory_safe(model: HyperbolicLCM, x: torch.Tensor) -> torch.Tensor:
    return model(x)


def hlcm_tangent_sequence(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)

    if cfg.strict_finite_checks:
        assert_finite("input_x_before_norm", x)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    x = sanitize_tensor(x, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h = model(x)
    h = sanitize_tensor(h, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)
    h_tan = model.manifold.logmap0(h)
    h_tan = sanitize_tensor(h_tan, cfg.clamp_tangent_value, cfg.replace_nonfinite_with_zero)

    if cfg.strict_finite_checks:
        assert_finite("hlcm_h_tan", h_tan)

    return h_tan


def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
) -> torch.Tensor:
    h_tan = hlcm_tangent_sequence(model, x, pad_mask, mu, sigma, cfg)
    pad_mask = pad_mask.to(h_tan.device, non_blocking=True)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, Any],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma, cfg=cfg)
    e_q = F.normalize(e_q, dim=-1)

    logits_list = []
    step = max(1, cfg.choice_chunk_size)

    for k0 in range(0, K, step):
        k1 = min(K, k0 + step)

        ch = choices[:, k0:k1]
        ch_mask = cmask[:, k0:k1]

        flat = ch.reshape(B * (k1 - k0), T, D)
        flat_mask = ch_mask.reshape(B * (k1 - k0), T)

        e_c = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma, cfg=cfg)
        e_c = e_c.reshape(B, (k1 - k0), -1)
        e_c = F.normalize(e_c, dim=-1)

        logits_list.append(torch.einsum("bd,bkd->bk", e_q, e_c))

    logits = torch.cat(logits_list, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)
    logits = logits.masked_fill(~choice_mask, torch.finfo(logits.dtype).min)
    return logits


# ============================================================
# METRICS
# ============================================================

def safe_div(a: float, b: float) -> float:
    return a / b if b else 0.0


def multiclass_classification_metrics(y_true: List[int], y_pred: List[int], num_classes: int) -> Dict[str, float]:
    out = {}
    supports = []

    macro_p = macro_r = macro_f1 = 0.0
    weighted_p = weighted_r = weighted_f1 = 0.0

    for cls in range(num_classes):
        tp = sum(t == cls and p == cls for t, p in zip(y_true, y_pred))
        fp = sum(t != cls and p == cls for t, p in zip(y_true, y_pred))
        fn = sum(t == cls and p != cls for t, p in zip(y_true, y_pred))
        support = sum(t == cls for t in y_true)
        supports.append(support)

        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall)

        out[f"precision_class_{cls}"] = precision
        out[f"recall_class_{cls}"] = recall
        out[f"f1_class_{cls}"] = f1
        out[f"support_class_{cls}"] = support

        macro_p += precision
        macro_r += recall
        macro_f1 += f1

    total_support = sum(supports)

    for cls in range(num_classes):
        w = safe_div(supports[cls], total_support)
        weighted_p += out[f"precision_class_{cls}"] * w
        weighted_r += out[f"recall_class_{cls}"] * w
        weighted_f1 += out[f"f1_class_{cls}"] * w

    out["precision_macro"] = macro_p / max(1, num_classes)
    out["recall_macro"] = macro_r / max(1, num_classes)
    out["f1_macro"] = macro_f1 / max(1, num_classes)

    out["precision_weighted"] = weighted_p
    out["recall_weighted"] = weighted_r
    out["f1_weighted"] = weighted_f1

    return out


def multiclass_brier_score(probs: torch.Tensor, labels: torch.Tensor, choice_mask: torch.Tensor) -> torch.Tensor:
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)
    sq_error = ((probs - one_hot) ** 2).masked_fill(~choice_mask, 0.0)
    return sq_error.sum(dim=1)


def expected_calibration_error(confidences: torch.Tensor, correctness: torch.Tensor, n_bins: int = 15) -> Tuple[float, float]:
    ece = 0.0
    mce = 0.0

    for b in range(n_bins):
        lo = b / n_bins
        hi = (b + 1) / n_bins

        if b == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()

            ece += mask.float().mean().item() * gap
            mce = max(mce, gap)

    return float(ece), float(mce)


@torch.no_grad()
def evaluate_after_sft(
    model: HyperbolicLCM,
    loader: DataLoader,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: EvalConfig,
    k_values=(1, 2, 3, 4, 5, 8),
) -> Tuple[Dict[str, float], List[Dict[str, Any]]]:
    model.eval()

    total = 0
    total_loss = 0.0
    total_nll = 0.0
    correct = 0
    total_chance = 0.0
    total_brier = 0.0

    y_true, y_pred = [], []
    all_confidences = []
    all_correctness = []

    rank_sums = {"mrr": 0.0}
    for k in k_values:
        rank_sums[f"precision@{k}"] = 0.0
        rank_sums[f"recall@{k}"] = 0.0

    predictions = []

    for batch in tqdm(loader, desc="Evaluating after_sft"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg)
        labels = batch["label"].to(logits.device, non_blocking=True)
        choice_mask = batch["choice_mask"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        log_probs = F.log_softmax(logits, dim=-1)
        probs = F.softmax(logits, dim=-1)

        bs = labels.size(0)
        total += bs
        total_loss += float(loss.item()) * bs
        total_nll += float((-log_probs.gather(1, labels.view(-1, 1)).squeeze(1)).sum().item())

        preds = logits.argmax(dim=1)
        batch_correct = preds == labels
        correct += int(batch_correct.sum().item())

        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(preds.detach().cpu().tolist())

        total_brier += float(multiclass_brier_score(probs, labels, choice_mask).sum().item())

        confidences = probs.max(dim=1).values
        all_confidences.append(confidences.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        ranked = torch.argsort(logits, dim=1, descending=True)
        choice_counts = choice_mask.sum(dim=1).detach().cpu().tolist()

        for i in range(bs):
            gold = int(labels[i].item())
            pred = int(preds[i].item())
            valid_k = int(choice_counts[i])
            total_chance += 1.0 / max(1, valid_k)

            ranked_i = ranked[i, :valid_k]
            rank_pos = (ranked_i == gold).nonzero(as_tuple=False)

            if rank_pos.numel() > 0:
                rank = int(rank_pos.item()) + 1
                rank_sums["mrr"] += 1.0 / rank

                for k in k_values:
                    kk = min(k, valid_k)
                    hit = 1.0 if gold in ranked_i[:kk].tolist() else 0.0
                    rank_sums[f"recall@{k}"] += hit
                    rank_sums[f"precision@{k}"] += hit / kk

            logits_i = logits[i, :valid_k].detach().cpu()
            probs_i = probs[i, :valid_k].detach().cpu()
            choices = batch["choice_texts"][i]

            predictions.append({
                "question_text": batch["question_texts"][i],
                "gold_answer": batch["gold_answers"][i],
                "pred_answer": choices[pred],
                "gold_choice_index": gold,
                "pred_choice_index": pred,
                "correct": int(pred == gold),
                "confidence": float(probs_i[pred].item()),
                "choice_texts": choices,
                "choice_logits": [float(x) for x in logits_i.tolist()],
                "choice_probs": [float(x) for x in probs_i.tolist()],
            })

    confidences_cat = torch.cat(all_confidences) if all_confidences else torch.empty(0)
    correctness_cat = torch.cat(all_correctness) if all_correctness else torch.empty(0)
    ece, mce = expected_calibration_error(confidences_cat, correctness_cat, n_bins=cfg.ece_bins)

    num_classes = max(cfg.num_choices, max(y_true + y_pred) + 1 if y_true else cfg.num_choices)

    metrics = {
        "num_examples": total,
        "loss": total_loss / max(1, total),
        "nll": total_nll / max(1, total),
        "accuracy": correct / max(1, total),
        "chance_accuracy": total_chance / max(1, total),
        "brier_score": total_brier / max(1, total),
        "ece": ece,
        "mce": mce,
        "ece_bins": cfg.ece_bins,
    }

    for key, val in rank_sums.items():
        metrics[key] = val / max(1, total)

    metrics.update(multiclass_classification_metrics(y_true, y_pred, num_classes=num_classes))
    return metrics, predictions


# ============================================================
# MAIN
# ============================================================

def main():
    set_seed(cfg.seed)
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("EVAL ONLY. No training.")
    print("Checkpoint:", cfg.eval_ckpt_path)
    print("Using normalizer:", cfg.use_normalizer)

    train_hf, eval_hf = load_gsm8k_local(cfg)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    if cfg.hard_negative_pool_from_train_only:
        answer_pool = collect_answer_pool(train_hf)
    else:
        answer_pool = sorted(set(collect_answer_pool(train_hf)).union(set(collect_answer_pool(eval_hf))))

    answer_pool_meta = build_answer_pool_metadata(answer_pool)
    print(f"[pool] unique answers in hard-negative pool: {len(answer_pool_meta)}")

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
        answer_pool_meta=answer_pool_meta,
    )

    del conceptizer
    cuda_cleanup()

    eval_loader = DataLoader(
        CachedMCQDataset(eval_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model, ckpt_info = load_base_then_after_sft(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device, cfg.use_normalizer)

    metrics, predictions = evaluate_after_sft(
        model=model,
        loader=eval_loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        k_values=(1, 2, 3, 4, 5, 8),
    )

    result = {
        "dataset": "gsm8k_local",
        "split": "validation/test parquet",
        "checkpoint": cfg.eval_ckpt_path,
        "checkpoint_stage": ckpt_info.get("stage", None),
        "checkpoint_sft_best_acc": ckpt_info.get("sft_best_acc", None),
        "metrics": metrics,
        "config": asdict(cfg),
    }

    metrics_path = os.path.join(cfg.out_dir, "after_sft_eval_precision_brier_ece.json")
    preds_path = os.path.join(cfg.out_dir, "after_sft_eval_predictions.json")

    safe_json_dump(result, metrics_path)
    safe_json_dump(predictions, preds_path)

    print("\n========== after_sft.pt EVAL ==========")
    print(f"Examples:        {metrics['num_examples']}")
    print(f"Loss:            {metrics['loss']:.6f}")
    print(f"NLL:             {metrics['nll']:.6f}")
    print(f"Accuracy:        {metrics['accuracy']:.6f}")
    print(f"Chance accuracy: {metrics['chance_accuracy']:.6f}")
    print(f"Precision macro: {metrics['precision_macro']:.6f}")
    print(f"Recall macro:    {metrics['recall_macro']:.6f}")
    print(f"F1 macro:        {metrics['f1_macro']:.6f}")
    print(f"Precision wtd:   {metrics['precision_weighted']:.6f}")
    print(f"Recall wtd:      {metrics['recall_weighted']:.6f}")
    print(f"F1 weighted:     {metrics['f1_weighted']:.6f}")
    print(f"MRR:             {metrics['mrr']:.6f}")
    print(f"Brier score:     {metrics['brier_score']:.6f}")
    print(f"ECE:             {metrics['ece']:.6f}")
    print(f"MCE:             {metrics['mce']:.6f}")

    print("\nSaved:")
    print(metrics_path)
    print(preds_path)

    del model
    cuda_cleanup()


if __name__ == "__main__":
    main()

/home/user/anaconda3/envs/raat/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Device: cpu
EVAL ONLY. No training.
Checkpoint: runs/hlcm_gsm8k/after_sft.pt
Using normalizer: False


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pool] unique answers in hard-negative pool: 866
[cache] building validation


cache:validation: 100%|█████████████████████████████████████████| 1319/1319 [05:20<00:00,  4.12it/s]


[cache] saved gsm8k_hardneg_cache/validation_hardneg_tok256_seq8_K8.pt (1319 examples, skipped=0)
[load] after_sft checkpoint: runs/hlcm_gsm8k/after_sft.pt
[load] stage: after_sft
[load] missing=0, unexpected=0


Evaluating after_sft: 100%|███████████████████████████████████████| 165/165 [13:56<00:00,  5.07s/it]



========== after_sft.pt EVAL ==========
Examples:        1319
Loss:            2.079419
NLL:             2.079419
Accuracy:        0.193328
Chance accuracy: 0.125000
Precision macro: 0.125000
Recall macro:    0.024166
F1 macro:        0.040502
Precision wtd:   1.000000
Recall wtd:      0.193328
F1 weighted:     0.324015
MRR:             0.398222
Brier score:     0.874994
ECE:             0.068296
MCE:             0.068296

Saved:
runs/hlcm_gsm8k/after_sft_eval_precision_brier_ece.json
runs/hlcm_gsm8k/after_sft_eval_predictions.json
